In [ ]:
# ============================================
# BLOOD CELL CLASSIFICATION (Colab) - kagglehub + Pretrained CNN (PyTorch EfficientNet-B0)
# Dataset: https://www.kaggle.com/datasets/unclesamulus/blood-cells-image-dataset
# ============================================

# 1) Install deps (run once)
!pip -q install kagglehub torch torchvision scikit-learn

# 2) Download dataset via kagglehub
import os
import kagglehub

path = kagglehub.dataset_download("unclesamulus/blood-cells-image-dataset")
print("Path to dataset files:", path)

# 3) Find the images folder automatically (robust)
def find_images_dir(base_path: str) -> str:
    """
    Finds a directory that looks like an ImageFolder root:
    it contains subfolders and those subfolders contain image files.
    """
    image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")
    candidates = []
    for root, dirs, files in os.walk(base_path):
        if not dirs:
            continue
        ok_subdirs = 0
        for d in dirs[:10]:
            sub = os.path.join(root, d)
            try:
                for f in os.listdir(sub):
                    if f.lower().endswith(image_exts):
                        ok_subdirs += 1
                        break
            except Exception:
                pass
        if ok_subdirs >= 2:
            candidates.append(root)

    if not candidates:
        raise FileNotFoundError(
            "Could not auto-detect the images folder. "
            "Please print os.listdir(path) and share the output."
        )

    candidates.sort(key=lambda p: (("images" not in p.lower()), len(p)))
    return candidates[0]

DATA_DIR = find_images_dir(path)
print("Detected DATA_DIR:", DATA_DIR)
print("Class folders:", os.listdir(DATA_DIR))

# 4) Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix

# 5) Config
# EfficientNet-B0 default input size is 224x224 (same as ResNet50)
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS_FROZEN   = 8       # stage 1: head only
EPOCHS_FINETUNE = 5       # stage 2: partial unfreeze
SEED        = 42
VAL_SPLIT   = 0.2
LR_FROZEN   = 1e-3
LR_FINETUNE = 1e-5
NUM_WORKERS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(SEED)

# 6) Transforms
# EfficientNet uses the same ImageNet mean/std as ResNet
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.08, contrast=0.12, saturation=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# 7) Dataset + split
full_ds = ImageFolder(DATA_DIR, transform=train_tfms)
class_names = full_ds.classes
num_classes = len(class_names)
print("Classes:", class_names)
print("Num classes:", num_classes)
print("Total images:", len(full_ds))

val_size   = int(VAL_SPLIT * len(full_ds))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

val_ds.dataset.transform = val_tfms

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# 8) Model: Pretrained EfficientNet-B0
# -----------------------------------------------
# EfficientNet-B0 architecture (torchvision):
#   model.features  — MBConv backbone (frozen in stage 1)
#   model.avgpool   — adaptive average pool
#   model.classifier[0] — Dropout
#   model.classifier[1] — Linear(1280, num_classes)  <-- we replace this
# -----------------------------------------------
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model   = torchvision.models.efficientnet_b0(weights=weights)

# Replace the final Linear layer to match our number of classes
# in_features for EfficientNet-B0 classifier is 1280
in_features = model.classifier[1].in_features          # 1280
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

print("EfficientNet-B0 classifier head:", model.classifier)

# 9) Helpers
def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss   = criterion(logits, y)

            if train:
                loss.backward()
                optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_from_logits(logits, y) * bs
        n += bs

    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate_and_report():
    model.eval()
    y_true, y_pred = [], []
    for x, y in val_loader:
        x      = x.to(device, non_blocking=True)
        logits = model(x)
        preds  = torch.argmax(logits, dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(y.numpy())

    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# 10) Stage 1: Freeze backbone (features), train classifier head only
# -----------------------------------------------
# EfficientNet uses model.features for the backbone
# and model.classifier for the head — freeze features, train classifier
# -----------------------------------------------
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LR_FROZEN)

print("\n=== STAGE 1: Training classifier head (backbone frozen) ===")
best_val_acc = 0.0
best_path    = "best_efficientnetb0_bloodcells.pth"

for epoch in range(EPOCHS_FROZEN):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    print(f"Epoch {epoch+1}/{EPOCHS_FROZEN} | "
          f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes":  class_names,
            "data_dir": DATA_DIR,
        }, best_path)

print("Best val acc (stage 1):", best_val_acc)

# 11) Stage 2: Fine-tune last MBConv block + classifier
# -----------------------------------------------
# EfficientNet-B0 has 9 sub-blocks inside model.features (indices 0–8).
# Unfreezing features[6], features[7], features[8] + classifier
# mirrors the "unfreeze last block" strategy used for layer4 in ResNet.
# -----------------------------------------------
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 3 MBConv blocks (features[6], [7], [8]) + classifier
UNFREEZE_FROM = 6   # tweak this if you want more/fewer blocks unfrozen
for idx in range(UNFREEZE_FROM, len(list(model.features))):
    for param in model.features[idx].parameters():
        param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params, lr=LR_FINETUNE)

print("\n=== STAGE 2: Fine-tuning (features[6-8] + classifier unfrozen) ===")
best_val_acc_ft = best_val_acc

for epoch in range(EPOCHS_FINETUNE):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    print(f"Epoch {epoch+1}/{EPOCHS_FINETUNE} | "
          f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

    if va_acc > best_val_acc_ft:
        best_val_acc_ft = va_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes":  class_names,
            "data_dir": DATA_DIR,
        }, best_path)

print("Best val acc (after fine-tune):", best_val_acc_ft)

# 12) Final evaluation report
model.load_state_dict(torch.load(best_path, map_location=device)["model_state_dict"])
evaluate_and_report()

# 13) Download checkpoint in Colab
from google.colab import files
files.download(best_path)

print("\nDONE ✅ Model saved as:", best_path)

Using Colab cache for faster access to the 'blood-cells-image-dataset' dataset.
Path to dataset files: /kaggle/input/blood-cells-image-dataset
Detected DATA_DIR: /kaggle/input/blood-cells-image-dataset/bloodcells_dataset
Class folders: ['monocyte', 'ig', 'neutrophil', 'basophil', 'lymphocyte', 'erythroblast', 'eosinophil', 'platelet']
Using device: cuda
Classes: ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']
Num classes: 8
Total images: 17092
EfficientNet-B0 classifier head: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=8, bias=True)
)

=== STAGE 1: Training classifier head (backbone frozen) ===
Epoch 1/8 | train_loss=0.9965 train_acc=0.6939 | val_loss=0.6218 val_acc=0.8174
Epoch 2/8 | train_loss=0.6514 train_acc=0.7912 | val_loss=0.5095 val_acc=0.8449
Epoch 3/8 | train_loss=0.5650 train_acc=0.8176 | val_loss=0.4667 val_acc=0.8552
Epoch 4/8 | train_loss=0.5314 train_acc=0.8241 | val_los